# 탐색적 분석 계획 — 사전 근거 없는 세그먼트 축 추가 검토

## 배경
H4(프로필 등록 여부)는 가설 트리에 사전 근거가 있어 확증적으로 검증했다. 아래 4개 축은 "왜 이 세그먼트가 문제일 것"이라는 이론적 근거를 사전에 세운 적이 없어, 확증적 가설과 분리해 탐색적으로만 살펴본다. 결과가 유의하더라도 "확정 결론"이 아니라 "후속 검증이 필요한 참고 신호"로 취급한다.

## 대상 축과 방법

| 축 | 변수 형태 | 순이익 검정 | 무임승차 검정 |
|---|---|---|---|
| income_quartile | 범주형(4구간) | Kruskal-Wallis H | 카이제곱(4×2) |
| membership_days | 연속형 | Spearman 상관 | Mann-Whitney U (무임승차 여부에 따른 가입경과일 차이) |
| age | 연속형 | Spearman 상관 | Mann-Whitney U (무임승차 여부에 따른 나이 차이) |
| gender | 범주형(M/F만, O는 표본 작아 제외) | Mann-Whitney U | 카이제곱(2×2) |

- profit은 여전히 왜도가 커서 모든 순이익 비교에 비모수 방법(Kruskal-Wallis, Spearman, Mann-Whitney) 사용
- age는 프로필 미등록 고객(2,175명)이 결측이라 해당 분석에서 자동 제외됨

## 다중비교 보정
4개 축 × 2개 지표 = 총 8개 검정을 한 분석 안에서 시도하는 것이므로, 개별 검정을 α=0.05로 각각 봐도 "8개 중 하나라도 우연히 유의하게 나올 확률"이 훨씬 커진다. 이를 막기 위해 Bonferroni 보정을 적용한다.

```
보정 유의수준 = 0.05 / 8 = 0.00625
```

이 스크립트의 모든 유의성 판정은 0.05가 아니라 0.00625를 기준으로 한다.

## 진행 순서
income_quartile → membership_days → age → gender

## 해석 시 유의할 점
표본 크기가 3만 건 이상으로 커서, 아주 작은 실질적 차이도 p-value가 극히 작게 나올 수 있다. 따라서 p-value만으로 판단하지 않고 효과크기(rank-biserial, Cramér's V, Spearman rho)를 반드시 함께 보고, "통계적으로 유의함"과 "실질적으로 의미 있음"을 구분해서 결론 내린다.

In [1]:
"""
탐색적 분석 스크립트
====================
사전 이론적 근거가 없어 확증적 가설(H4)에는 넣지 않았던 4개 세그먼트 축을
추가로 살펴보는 스크립트.

대상 축: income_quartile, membership_days, age, gender
지표: ① 완료당 순이익(profit)  ② 엄격 기준 무임승차 비율(freeride)

무임승차(freeride) 정의:
  - is_viewed == 0 (아예 안 봄), 또는
  - viewed_time > completed_time (봤지만 완료 이후에 봄, 즉 완료에 영향 못 줌)
  위 둘 중 하나면 True, 완료 이전에 정상적으로 열람했으면 False

다중비교 보정:
  4개 축 x 2개 지표 = 총 8개 검정 -> Bonferroni 보정 유의수준 = 0.05 / 8 = 0.00625
  (이 스크립트의 모든 판정은 0.05가 아니라 이 보정값 기준으로 함)
"""

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

SEP = "=" * 70
ALPHA_RAW = 0.05
N_TESTS = 8
ALPHA_CORRECTED = ALPHA_RAW / N_TESTS


def section(title):
    print(f"\n{SEP}\n{title}\n{SEP}")


def sig_mark(p):
    return f"(보정 후 {'유의함' if p < ALPHA_CORRECTED else '유의하지 않음'}, 기준={ALPHA_CORRECTED:.5f})"


# =========================================================
# 0. 데이터 로드
# =========================================================
section("STEP 0. 데이터 로드 및 파생 변수 준비")

df = pd.read_csv("offer_instance_table.csv")
completed = df[df["is_completed"] == 1].copy()
completed = completed[completed["matched_amount"].notna()]
completed["profit"] = completed["matched_amount"] - completed["reward"]


def is_freeride(row):
    if row["is_viewed"] == 0:
        return True
    if row["viewed_time"] > row["completed_time"]:
        return True
    return False


completed["freeride"] = completed.apply(is_freeride, axis=1)
print(f"완료 인스턴스 수: {len(completed):,}건")
print(f"Bonferroni 보정 유의수준: 0.05 / {N_TESTS} = {ALPHA_CORRECTED:.5f}")


def mannwhitney_report(g0, g1, label, name0="그룹0", name1="그룹1"):
    u, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    n0, n1 = len(g0), len(g1)
    rbc = 1 - (2 * u) / (n0 * n1)
    print(f"\n[{label}]")
    print(f"  {name0}: n={n0:,}, median={g0.median():.2f}, mean={g0.mean():.2f}")
    print(f"  {name1}: n={n1:,}, median={g1.median():.2f}, mean={g1.mean():.2f}")
    print(f"  Mann-Whitney U={u:,.0f}, p={p:.6f}  {sig_mark(p)}")
    print(f"  rank-biserial r={rbc:.4f}")
    return p, rbc


def chi2_report(ct, label):
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"\n[{label}]")
    print(ct)
    print(f"  카이제곱={chi2:.2f}, p={p:.6f}  {sig_mark(p)}")
    print(f"  Cramér's V={cramers_v:.4f}")
    min_expected = expected.min()
    print(f"  최소 기대빈도={min_expected:.2f} "
          f"({'가정 충족' if min_expected >= 5 else '가정 위반! 주의'})")
    return p, cramers_v


# =========================================================
# 축 1. income_quartile
# =========================================================
section("탐색1. income_quartile - 완료당 순이익")

groups = [completed[completed["income_quartile"] == q]["profit"]
          for q in ["Q1_저", "Q2_중하", "Q3_중상", "Q4_고"]]
h_stat, p_kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis H={h_stat:.2f}, p={p_kw:.6f}  {sig_mark(p_kw)}")
for q, g in zip(["Q1_저", "Q2_중하", "Q3_중상", "Q4_고"], groups):
    print(f"  {q}: n={len(g):,}, median={g.median():.2f}")

section("탐색1. income_quartile - 무임승차 비율")
ct = pd.crosstab(completed["income_quartile"], completed["freeride"])
ct.columns = ["정상완료", "무임승차"]
chi2_report(ct, "income_quartile별 무임승차")


# =========================================================
# 축 2. membership_days (연속형)
# =========================================================
section("탐색2. membership_days - 완료당 순이익 (Spearman 상관)")

sub = completed.dropna(subset=["membership_days", "profit"])
rho, p_rho = stats.spearmanr(sub["membership_days"], sub["profit"])
print(f"n={len(sub):,}")
print(f"Spearman rho={rho:.4f}, p={p_rho:.6f}  {sig_mark(p_rho)}")

section("탐색2. membership_days - 무임승차 여부에 따른 가입경과일 차이")
g0 = completed[completed["freeride"] == False]["membership_days"].dropna()
g1 = completed[completed["freeride"] == True]["membership_days"].dropna()
mannwhitney_report(g0, g1, "정상완료 vs 무임승차 - membership_days",
                    name0="정상완료", name1="무임승차")


# =========================================================
# 축 3. age (연속형, 미등록 고객은 결측이라 자동 제외됨)
# =========================================================
section("탐색3. age - 완료당 순이익 (Spearman 상관)")

sub = completed.dropna(subset=["age", "profit"])
print(f"age 결측 제외 후 n={len(sub):,} (전체 {len(completed):,}건 중 "
      f"{len(completed)-len(sub):,}건은 age 결측이라 제외 - 대부분 미등록 고객)")
rho, p_rho = stats.spearmanr(sub["age"], sub["profit"])
print(f"Spearman rho={rho:.4f}, p={p_rho:.6f}  {sig_mark(p_rho)}")

section("탐색3. age - 무임승차 여부에 따른 나이 차이")
g0 = completed[completed["freeride"] == False]["age"].dropna()
g1 = completed[completed["freeride"] == True]["age"].dropna()
mannwhitney_report(g0, g1, "정상완료 vs 무임승차 - age",
                    name0="정상완료", name1="무임승차")


# =========================================================
# 축 4. gender (M/F만 비교, O는 표본 작아 제외)
# =========================================================
section("탐색4. gender - 완료당 순이익 (M vs F)")

sub = completed[completed["gender"].isin(["M", "F"])]
print(f"gender 분포(완료건 기준): {dict(sub['gender'].value_counts())}")
g_m = sub[sub["gender"] == "M"]["profit"]
g_f = sub[sub["gender"] == "F"]["profit"]
mannwhitney_report(g_m, g_f, "M vs F - profit", name0="M", name1="F")

section("탐색4. gender - 무임승차 비율 (M vs F)")
ct = pd.crosstab(sub["gender"], sub["freeride"])
ct.columns = ["정상완료", "무임승차"]
chi2_report(ct, "gender별 무임승차 (M/F만)")


# =========================================================
# 최종 요약
# =========================================================
section("탐색적 분석 전체 요약")
print(f"""
총 {N_TESTS}개 검정 수행, Bonferroni 보정 유의수준 = {ALPHA_CORRECTED:.5f}

이 결과들은 사전 가설 없이 추가로 살펴본 탐색적 발견이며,
유의하게 나온 항목이 있더라도 "확정 결론"이 아니라 "후속 검증이 필요한 참고 신호"로
발표에 표시해야 함.
""")


STEP 0. 데이터 로드 및 파생 변수 준비
완료 인스턴스 수: 33,101건
Bonferroni 보정 유의수준: 0.05 / 8 = 0.00625

탐색1. income_quartile - 완료당 순이익
Kruskal-Wallis H=11794.83, p=0.000000  (보정 후 유의함, 기준=0.00625)
  Q1_저: n=5,965, median=4.75
  Q2_중하: n=7,944, median=8.41
  Q3_중상: n=8,416, median=13.12
  Q4_고: n=9,675, median=19.58

탐색1. income_quartile - 무임승차 비율

[income_quartile별 무임승차]
                 정상완료  무임승차
income_quartile            
Q1_저             4053  1912
Q2_중하            5872  2072
Q3_중상            5964  2452
Q4_고             6445  3230
  카이제곱=124.69, p=0.000000  (보정 후 유의함, 기준=0.00625)
  Cramér's V=0.0624
  최소 기대빈도=1801.80 (가정 충족)

탐색2. membership_days - 완료당 순이익 (Spearman 상관)
n=33,101
Spearman rho=-0.0205, p=0.000194  (보정 후 유의함, 기준=0.00625)

탐색2. membership_days - 무임승차 여부에 따른 가입경과일 차이

[정상완료 vs 무임승차 - membership_days]
  정상완료: n=23,267, median=535.00, mean=598.64
  무임승차: n=9,834, median=511.00, mean=573.42
  Mann-Whitney U=117,812,564, p=0.000018  (보정 후 유의함, 기준=0.00625)
  rank-biserial r=-0.0298

탐색3. age

# 탐색적 분석 결과 (쉬운 버전)

> 사전 근거 없이 추가로 살펴본 4개 축. 확정 결론이 아니라 "다음에 뭘 검증할지"를 위한 참고 자료.
> 보정 유의수준: 0.05 / 8 = 0.00625 (전부 통과했으나, 표본이 커서 p값은 원래 작게 나오기 쉬움 → 효과크기로 판단)

## 결과 한눈에

| 축 | 순이익과의 관계 | 무임승차와의 관계 |
|---|---|---|
| **age** | ρ=0.23 (중간) ⭐ 가장 뚜렷 | 거의 없음 |
| **income** | 계단식 증가, 구간별로 다 다름 ⭐ 뚜렷 | 작음 |
| gender | 여성이 다소 높음(작음~중간) | 작음 |
| membership_days | 사실상 없음 | 사실상 없음 |

## 정리

- **age, income**: 의미 있는 신호. 다음 분석에서 정식(확증적)으로 다뤄볼 가치 있음
- **gender**: 약한 신호. 다만 age·income과 겹쳐서 나온 착시일 수 있어 단독으로 믿기는 이름
- **membership_days**: 효과 없음. "오래된 고객일수록 문제"라는 추측은 틀렸음 → 더 안 봐도 됨

## 다음 액션 제안

1. **age를 다음 프로젝트의 확증적 가설 후보 1순위로**
2. **income도 후보로 유지** (이번엔 범위에서 뺐지만 신호가 뚜렷함)
3. **gender는 age·income을 통제한 뒤에도 효과가 남는지 확인 필요** (지금은 근거 부족)
4. **membership_days는 폐기**